In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

In [2]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.7 MB/s eta 0:00:00


In [3]:
df = pd.read_csv("/content/drive/MyDrive/ALL_DATASET/Time_series/train_updated_SKU.csv")
df.head(20)

,date,store,item,sales
0,01-01-2013,1,1,13
1,02-01-2013,1,1,11
2,03-01-2013,1,1,14
3,04-01-2013,1,1,13
4,05-01-2013,1,1,10
5,06-01-2013,1,1,12
6,07-01-2013,1,1,10
7,08-01-2013,1,1,9
8,09-01-2013,1,1,12
9,10-01-2013,1,1,9


In [4]:
df['item'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50])

In [5]:
df.isna().sum()

,0
date,0
store,0
item,0
sales,0


In [6]:
# STEP 1: DATA INGESTION & CONTINUOUS TIME FIX
print("Step 1: Loading and fixing timeline gaps...")

# Assuming your data is in a CSV. Replace with your actual file path.
# df = pd.read_csv('train.csv')

# For demonstration, let's assume 'df' is already loaded with columns: ['date', 'store', 'item', 'sales']
df['date'] = pd.to_datetime(df['date'], format="%d-%m-%Y")

def make_continuous_and_fill(group):
    """
    Business Logic: Retail data often omits days with zero sales.
    If we don't explicitly fill these missing days with 0, our lag features
    will calculate incorrect time distances.
    """
    # Find the min and max date for this specific store-item combination
    full_range = pd.date_range(start=group['date'].min(), end=group['date'].max(), freq='D')

    # Set date as index, reindex to the full continuous range, and fill missing sales with 0
    group = group.set_index('date').reindex(full_range)
    group['sales'] = group['sales'].fillna(0)

    # Bring the date index back as a column
    group = group.rename_axis('date').reset_index()
    return group

# Apply the continuous time fix across all stores and items
# Note: groupby operations can be slow on massive datasets, but are necessary for data integrity.
df_continuous = df.groupby(['store', 'item']).apply(make_continuous_and_fill).reset_index(drop=True)

# Forward fill store and item columns which became NaN during reindexing
df_continuous['store'] = df_continuous.groupby('item')['store'].ffill().bfill()
df_continuous['item'] = df_continuous.groupby('store')['item'].ffill().bfill()

Step 1: Loading and fixing timeline gaps...


/tmp/ipykernel_9058/1505848347.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_continuous = df.groupby(['store', 'item']).apply(make_continuous_and_fill).reset_index(drop=True)


In [7]:
df.head(20)

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10
5,2013-01-06,1,1,12
6,2013-01-07,1,1,10
7,2013-01-08,1,1,9
8,2013-01-09,1,1,12
9,2013-01-10,1,1,9


In [8]:
df

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10
...,...,...,...,...
912995,2017-12-27,10,50,63
912996,2017-12-28,10,50,59
912997,2017-12-29,10,50,74
912998,2017-12-30,10,50,62


In [9]:

print("Step 2: Engineering retail-specific features...")

# 2A. Calendar Features (Capturing Seasonality)
df_continuous['day_of_week'] = df_continuous['date'].dt.dayofweek
df_continuous['month'] = df_continuous['date'].dt.month
df_continuous['year'] = df_continuous['date'].dt.year

# Sort chronologically before calculating lags to prevent data leakage
df_continuous = df_continuous.sort_values(by=['store', 'item', 'date'])

# 2B. Lag Features (Capturing Momentum & Weekly Cycles)
# We calculate lags WITHIN each store-item group so Store A's sales don't bleed into Store B's lag.
df_continuous['lag_1'] = df_continuous.groupby(['store', 'item'])['sales'].shift(1)
df_continuous['lag_7'] = df_continuous.groupby(['store', 'item'])['sales'].shift(7)

# 2C. Rolling Features (Capturing Smoothed Trends)
# A 7-day rolling mean smooths out the daily noise (like weekend spikes)
df_continuous['rolling_7_mean'] = df_continuous.groupby(['store', 'item'])['sales'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
)

# Drop the rows where lag features are NaN (the first 7 days of the dataset)
df_features = df_continuous.dropna().copy()

Step 2: Engineering retail-specific features...


In [10]:
df_continuous.head()

,date,store,item,sales,day_of_week,month,year,lag_1,lag_7,rolling_7_mean
0,2013-01-01,1,1,13,1,1,2013,NaN,NaN,NaN
1,2013-01-02,1,1,11,2,1,2013,13.0,NaN,13.000000
2,2013-01-03,1,1,14,3,1,2013,11.0,NaN,12.000000
3,2013-01-04,1,1,13,4,1,2013,14.0,NaN,12.666667
4,2013-01-05,1,1,10,5,1,2013,13.0,NaN,12.750000


In [11]:
# STEP 3: CORRECTING CATEGORICAL VARIABLES

print("Step 3: Casting IDs to categorical types...")

# Statistically Sound Fix: Store 5 is NOT mathematically greater than Store 1.
# They are independent entities. We must cast them as pandas categories.
df_features['store'] = df_features['store'].astype('category')
df_features['item'] = df_features['item'].astype('category')

Step 3: Casting IDs to categorical types...


In [12]:
# STEP 4: OUT-OF-TIME (OOT) VALIDATION SPLIT

print("Step 4: Splitting data chronologically...")

# Never use train_test_split (random shuffling) on time-series data.
# We will train on everything before 2017, and test on 2017 to simulate real-world forecasting.

cutoff_date = '2017-01-01'
train_mask = df_features['date'] < cutoff_date
test_mask = df_features['date'] >= cutoff_date

# Define our features (X) and target (y)
features = ['store', 'item', 'day_of_week', 'month', 'year', 'lag_1', 'lag_7', 'rolling_7_mean']
target = 'sales'

X_train = df_features.loc[train_mask, features]
y_train = df_features.loc[train_mask, target]

X_test = df_features.loc[test_mask, features]
y_test = df_features.loc[test_mask, target]

Step 4: Splitting data chronologically...


In [ ]:
import optuna
import xgboost as xgb
import numpy as np

# STEP 4.5: BAYESIAN HYPERPARAMETER TUNING


print("Starting Optuna Study to find perfect hyperparameters...")

def objective(trial):
    """
    This is the function Optuna will try to minimize.
    It will pass different 'trial' values in here and check the final WAPE.
    """

    # 1. Define the "Search Space"
    param = {
        'enable_categorical': True,
        'n_jobs': -1,
        'random_state': 42,

        # We give it a massive maximum tree count
        'n_estimators': 1000,

        # MOVED HERE: early stopping is now declared during initialization!
        'early_stopping_rounds': 30,

        # Optuna will pick a float between 0.01 and 0.2
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        # Optuna will pick an integer between 3 and 10
        'max_depth': trial.suggest_int('max_depth', 3, 10),

        # 'subsample' prevents overfitting by only using a random % of rows per tree
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),

        # 'colsample_bytree' prevents overfitting by only using a random % of columns per tree
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }

    # 2. Build the model with the current trial's parameters
    model = xgb.XGBRegressor(**param)

    # 3. Train the model using our Out-of-Time validation set
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)], # Validation set for early stopping to monitor
        verbose=False                # Turn off printing so it doesn't flood our screen
        # Notice early_stopping_rounds is GONE from here!
    )

    # 4. Evaluate how well these parameters performed
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, a_min=0, a_max=None)

    # Calculate WAPE
    wape = np.sum(np.abs(y_test.values - y_pred)) / np.sum(y_test.values)

    return wape


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

# 6. Print the absolute best results
print("\n" + "="*40)
print(f"BEST WAPE SCORE ACHIEVED: {study.best_value:.2%}")
print("BEST HYPERPARAMETERS FOUND:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print("="*40)

[I 2026-03-31 16:16:33,284] A new study created in memory with name: no-name-842958b1-81c7-4d18-a520-9ea9df6e14a5


Starting Optuna Study to find perfect hyperparameters...


[I 2026-03-31 16:18:37,484] Trial 0 finished with value: 0.10784413098609695 and parameters: {'learning_rate': 0.016835513092459932, 'max_depth': 5, 'subsample': 0.7740771089393939, 'colsample_bytree': 0.7553056660908166}. Best is trial 0 with value: 0.10784413098609695.


In [ ]:
pip install --upgrade xgboost

In [ ]:


# STEP 5: MODEL TRAINING (GLOBAL PANEL DATA)

print("Step 5: Training the global XGBoost model...")

# Initialize XGBoost.
# enable_categorical=True is REQUIRED so XGBoost natively handles the 'store' and 'item' categories.
model = xgb.XGBRegressor(
    n_estimators=500,        # Number of trees
    learning_rate=0.05,      # Step size shrinkage
    max_depth=6,             # Tree depth
    enable_categorical=True, # The crucial fix for our IDs
    random_state=42,
    n_jobs=-1                # Use all CPU cores
)

# Fit the model
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    early_stopping_rounds=50, # Stop if validation score doesn't improve for 50 rounds
    verbose=50                # Print progress every 50 rounds
)


In [ ]:












# ==========================================
# STEP 6: EVALUATION (BUSINESS METRIC: WAPE)
# ==========================================
print("Step 6: Evaluating business impact...")

def calculate_wape(y_true, y_pred):
    """
    Weighted Absolute Percentage Error.
    Why WAPE? MAPE explodes when actual sales are 0. WAPE handles zeros gracefully
    and weights the error by the total volume of sales, making it highly interpretable
    for Supply Chain Managers setting safety stock limits.
    """
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

# Generate predictions for the unseen 2017 test data
y_pred = model.predict(X_test)

# Ensure predictions are not negative (you can't sell negative items)
y_pred = np.clip(y_pred, a_min=0, a_max=None)

wape_score = calculate_wape(y_test.values, y_pred)
mae_score = mean_absolute_error(y_test.values, y_pred)

print("-" * 30)
print(f"Final WAPE Score: {wape_score:.2%}")
print(f"Final MAE Score:  {mae_score:.2f} units")
print("-" * 30)

# Optional: Feature Importance to explain what drives demand
importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importances:")
print(importance)